# Feature distributions - Signal vs. background

In [1]:
from pathlib import Path
from diquark.config.config_manager import ConfigManager
from diquark.data.loader import DataLoader

config_files_dir = Path(".").absolute().parent / "diquark" / "config"

config_file_path = (
    # config_files_dir / "New_Features" / "ATLAS_136_S8000_B7500_32j_5f.yaml"
    # config_files_dir / "MadGraph_Data" / "MSuu_8000_shatmin_7500.yaml"
    config_files_dir / "MadGraph_Data" / "MSuu_8000_shatmin_7500_htjmin_2000.yaml"
)
assert config_file_path.exists()

config = ConfigManager(str(config_file_path))

# Load backgrounds config file
backgrounds_config_file = config.get_required("data.backgrounds_file")
backgrounds_config_file_path = (
    config_files_dir / "Backgrounds" / backgrounds_config_file
)
backgrounds_config = ConfigManager(backgrounds_config_file_path)

backgrounds_file_names = backgrounds_config.get("backgrounds.file_names", {})
backgrounds_file_name_mapping = backgrounds_config.get(
    "backgrounds.file_name_mapping", {}
)

backgrounds_directory = Path(
    backgrounds_config.get("backgrounds.base_directory", "/")
)

files: set[Path] = set()


def check_root_file_exists(path_str: str):
    path = Path(path_str)
    if not path.exists():
        raise Exception(f"Could not find background file at path '{path}'")
    files.add(path)
    return path


background_path_dict = {
    f"BKG:{key}": check_root_file_exists(path)
    for key, path in backgrounds_file_names.items()
}
assert len(files) == len(background_path_dict.keys())

def find_root_file_by_name(filename: str) -> str:
    results = backgrounds_directory.glob(f"*{filename}*.root")
    for path in results:
        files.add(path)
        return str(path)
    else:
        raise Exception(f"Could not find background file with filename '{filename}'")


background_path_dict |= {
    f"BKG:{key}": find_root_file_by_name(filename)
    for key, filename in backgrounds_file_name_mapping.items()
}
assert len(files) == len(background_path_dict.keys())

# Load signal config file
signal_config_file = config.get_required("data.signal_file")
backgrounds_config_file_path = config_files_dir / "Signals" / signal_config_file
signal_config = ConfigManager(backgrounds_config_file_path)

signal_file = Path(signal_config.get("signal.file"))
path_dict = background_path_dict | {"SIG:Suu": str(signal_file)}

data_loader = DataLoader(
    path_dict,
    index_start=0,
    index_stop=50_000,
)

In [2]:
data = data_loader.load_data()

Loading data:   0%|          | 0/7 [00:00<?, ?it/s]

In [3]:
from tqdm.contrib.concurrent import thread_map
from diquark.features.new_features import NewFeatureExtractor

print("Extracting features...")
feature_extractor = NewFeatureExtractor(max_jets=32, chi_mass=2000, suu_mass=8000)

features = thread_map(
    feature_extractor.compute_all,
    data.values(),
    max_workers=32,
    desc="Extracting features",
)

print(f"Working with {len(features[0].keys())} feature columns")

assert len(features[0].keys()) == len(feature_extractor.feature_names), (
    f"Number of extracted features ({len(features[0].keys())}) doesn't match number of feature names defined on feature extractor object ({len(feature_extractor.feature_names)})"
)

datasets = dict(zip(data.keys(), features))

Extracting features...


Extracting features:   0%|          | 0/7 [00:00<?, ?it/s]

Working with 95 feature columns


In [4]:
from diquark.data.preprocessor import Preprocessor

preprocessor = Preprocessor(config.get("preprocessing", {}))

In [5]:
df = preprocessor.create_dataframe(datasets)

In [6]:
df.head()

,jet_multiplicity,p_T_sum,p_T_min,p_T_mean,p_T_stddev,p_T_max,p_t_jet_1,p_t_jet_2,p_t_jet_3,p_t_jet_4,...,chi2_second_component_mean,chi2_second_component_stddev,chi2_second_component_max,chi2_third_component_sum,chi2_third_component_min,chi2_third_component_mean,chi2_third_component_stddev,chi2_third_component_max,Truth,target
0,7,2221.666260,26.014879,317.380894,253.699506,774.539551,774.539551,526.013611,365.332062,351.539581,...,1435.477455,2654.256337,9594.049805,3390.041748,56.677372,484.291678,605.850936,1597.370483,BKG:qcd,0
1,5,2799.837891,48.735893,559.967578,520.114902,1414.217163,1414.217163,856.100769,426.866913,53.917114,...,5360.004687,5285.219934,18486.816406,0.000000,-1.000000,-1.000000,-1.000000,-1.000000,BKG:qcd,0
2,4,2437.341797,58.883804,609.335449,544.282726,1242.210693,1242.210693,1057.025513,79.221741,58.883804,...,6208.072266,5149.006773,14337.033203,0.000000,-1.000000,-1.000000,-1.000000,-1.000000,BKG:qcd,0
3,7,2232.294678,25.960522,318.899240,364.008791,982.832581,982.832581,777.695496,226.926941,82.066643,...,2623.640848,4057.072317,18165.726562,2804.229492,0.055706,400.604213,721.368476,2125.776611,BKG:qcd,0
4,6,2315.158936,91.057823,385.859823,313.672848,1060.622314,1060.622314,351.299957,339.674622,240.661545,...,3184.692187,3795.786252,13796.495117,9.195731,9.195731,9.195731,NaN,9.195731,BKG:qcd,0


In [ ]:
import matplotlib.pyplot as plt

%config InlineBackend.figure_format = 'retina'

In [ ]:
plt.hist(
    df.sphericity[(df.target == 0)],
    bins=50,
    label="Background",
    alpha=0.5,
    density=True,
)
plt.hist(
    df.sphericity[(df.target == 1)],
    bins=50,
    label="Signal",
    alpha=0.5,
    density=True,
)
plt.xlabel("Sphericity")
plt.ylabel("Frequency")
plt.xlim(0, 0.5)
plt.title("Distribution of Sphericity")
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.hist(
    df.chi2_second_component_mean[df.target == 0],
    bins=90,
    label="Background",
    alpha=0.5,
    density=True,
)
plt.hist(
    df.chi2_second_component_mean[df.target == 1],
    bins=90,
    label="Signal",
    alpha=0.5,
    density=True,
)
plt.xlabel("Chi^2 coefficient")
plt.ylabel("Frequency")
plt.title("Chi^2 score for m3j vs. vectorlike boson mass")
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.hist(df.jet_multiplicity[df.target == 0], bins=50, label="Background", alpha=0.5, density=True)
plt.hist(df.jet_multiplicity[df.target == 1], bins=50, label="Signal", alpha=0.5, density=True)
plt.xlabel("Jet Multiplicity")
plt.ylabel("Frequency")
plt.title("Jet Multiplicity for Signal vs. Background")
plt.legend()
plt.grid()
plt.show()